In [15]:
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/NUIN/CBEM_pytorch
    # !pip install -r requirements.txt
    # Uninstall CPU jax/jaxlib if present
    # !pip uninstall -y jax jaxlib

    # # Install GPU-enabled JAX (choose the right cuda extra for your Colab)
    # # Try cuda12 first (common in Colab recently):
    # !pip install -U "jax[cuda12]"


## Stim

In [20]:
## Training data
data        = np.load("Data/PinkNoise_WholeCell_Light_beta_0_OFF_transient_medium_RF_032025Bc3_nr.npz", allow_pickle=True)
stim_mat        = data["stim"]
spk_t      = data['spk_times']
spk_train_mat   = data['spk_train']
V_train  = data['raw_traces'] # shape (N_train, T_v) 

## Test Data
data_test   = np.load("Data/PinkNoise_WholeCell_Light_beta_0_OFF_transient_medium_RF_032025Bc3_rs.npz", allow_pickle=True)
stim_rs        = data_test["stim"]
stimes_rs      = data_test['spk_times']
spk_t_test       = data_test['spk_train']
V_test  = data_test['raw_traces'] # shape (N_test,  T_v)
psth           = np.mean(spk_t_test, axis=0)

In [26]:

# Work entirely in ms
spk_t_ms = [np.asarray(tr) * 1000.0 for tr in spk_t]
dt_ms = 0.1
dt_stim_ms = 1000.0 / 60.0

# 0.5 s trimming on each side
presamples = int(round(500.0 / dt_ms))
postsamples = int(round(500.0 / dt_ms))

total_duration_ms = float(np.max([np.max(tr) if len(tr) else 0.0 for tr in spk_t_ms]))
num_bins = int(np.ceil(total_duration_ms / dt_ms))
bin_edges = np.linspace(0.0, num_bins * dt_ms, num_bins + 1)

def bin_one_trial(spike_times_ms, edges_ms):
    spikes = np.asarray(spike_times_ms, dtype=np.float32)
    counts, _ = np.histogram(spikes, bins=edges_ms)
    return counts

def upsample_hold_edges(I_frame, dt_stim_ms, dt_cell_ms, T_cell):
    t_cell = np.arange(T_cell) * dt_cell_ms
    idx = np.floor(t_cell / dt_stim_ms).astype(np.int32)
    idx = np.clip(idx, 0, I_frame.shape[-1] - 1)
    return I_frame[..., idx]

spk_train_full = np.stack(
    [bin_one_trial(spk_t_ms[trial], bin_edges) for trial in range(len(spk_t_ms))],
    axis=0
)

T_cell_full = spk_train_full.shape[1]
stimulus_full = upsample_hold_edges(stim_mat, dt_stim_ms=dt_stim_ms, dt_cell_ms=dt_ms, T_cell=T_cell_full)

spk_train = spk_train_full[:, presamples:-postsamples]
stimulus_cell = stimulus_full[:, presamples:-postsamples]

print(spk_train.shape, stimulus_cell.shape)



(11, 99853) (11, 99853)


In [109]:
nLinearRFs = 10 #;     %number of stimulus basis functions
numShortSHFilts = 5#; %each is 0.4ms long - used for refractory period
nSHfilters = 7+numShortSHFilts#;%7+

nInh = 1#;
nExc = 1#

RFstart = 2e-3 #;%0?
RFend   = 150e-3#;%150e-3 or 180e-3;
b = 0.02
t, B_orth, B_raw = makeRaisedCosBasis(nLinearRFs, dt, [RFstart, RFend], b, zflag=0)
B_orth.shape

(2478, 10)